In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeKyiv
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()

# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend=backend,optimization_level=2)

In [3]:


nb_qubits = 4

N = 2**nb_qubits
m = np.zeros((N,N))
for j in range(N):
    if j == N-1:
        break
    else:
       m[j,j+1] = -1 

for j in range(N):
    if j == N-1:
        break
    else:
       m[j+1,j] = -1 
for j in range(N):
   m[j,j] = 2 
m[0] = np.array([1]+ [0]*(N-1))
m[1,0] = 0

b = 0.25*np.array([0,1,1,1,1,1,1,1,1,1,1,1,1,2,0,0])
nb_qubits = 4

In [4]:
d,u = np.linalg.eig(m)

In [5]:
max(d)/min(d)

np.float64(103.08686891981992)

In [6]:
def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))
b

array([ 1.31838984e-16+1.38777878e-16j,  2.50000000e-01+2.15105711e-15j,
        2.50000000e-01+1.76247905e-15j,  2.50000000e-01+1.79023463e-15j,
        2.50000000e-01-1.15185639e-15j,  2.50000000e-01-1.38777878e-15j,
        2.50000000e-01-1.52655666e-15j,  2.50000000e-01-1.83186799e-15j,
        2.50000000e-01-2.12330153e-15j,  2.50000000e-01-1.91513472e-15j,
        2.50000000e-01-1.63757896e-15j,  2.50000000e-01-1.40165657e-15j,
        2.50000000e-01+1.24900090e-15j,  5.00000000e-01+2.84494650e-15j,
        1.84889275e-32+1.66533454e-16j, -3.69778549e-32+9.24446373e-33j])

In [7]:
def Hamiltonian(m):
    Ub = np.array(Operator(U_b(nb_qubits)))
    z = np.array([[1,0],
                 [0,-1]])
    I = np.array([[1,0],
                 [0,1]])
    def tensor(i,j,k,l):
        return np.kron(i,np.kron(j,np.kron(k,l)))
    M1 = (np.dot(np.dot(Ub,tensor(z,I,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,z,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,I,z,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,I,I,z)),np.conj(Ub.T)))
    M = 0.5*np.dot(np.dot(np.conj(m.T),(tensor(I,I,I,I) - M1/nb_qubits)),m)

    return M
A = Hamiltonian(m)

In [8]:

U1=scl.expm(2**0*2*np.pi*1j*A) 
U2=scl.expm(2**1*2*np.pi*1j*A) 
U3=scl.expm(2**2*2*np.pi*1j*A) 
U4=scl.expm(2**3*2*np.pi*1j*A) 
U5=scl.expm(2**4*2*np.pi*1j*A) 
U6=scl.expm(2**5*2*np.pi*1j*A) 
U7=scl.expm(2**6*2*np.pi*1j*A) 
U8=scl.expm(2**7*2*np.pi*1j*A)
 
u1gate = UnitaryGate(U1)
u2gate = UnitaryGate(U2)
u3gate=UnitaryGate(U3)
u4gate=UnitaryGate(U4)
u5gate=UnitaryGate(U5)
u6gate=UnitaryGate(U6)
u7gate=UnitaryGate(U7)
u8gate=UnitaryGate(U8)

C_u1gate=u1gate.control()
C_u2gate=u2gate.control()
C_u3gate=u3gate.control()
C_u4gate=u4gate.control()
C_u5gate=u5gate.control()
C_u6gate=u6gate.control()
C_u7gate=u7gate.control()
C_u8gate=u8gate.control()
    

In [9]:
nb_qubits = 4
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)
x_exact = np.linalg.solve(m,b)
x_exact = x_exact/np.linalg.norm(x_exact)
RMSE = []
rep = 1000
for _ in range(rep):
    Parameters = np.array([random.random() for _ in range(0, nb_params)])
    
    def ansatz(Parameters):
        qc = QuantumCircuit(N)
        for d in range(depth):
            param1=Parameters[d*9*N:(d+1)*(9*N)]
            for q in range(N):
                qc.ry(param1[q],qubits[q])
                qc.ry(param1[q+N],qubits[q])
                qc.ry(param1[q+2*N],qubits[q])
            qc.barrier()
            for q in range(N):
                qc.cx(qubits[q], qubits[(q+1)% N])
                qc.ry(param1[q+3*N],qubits[q])
                qc.ry(param1[q+4*N],qubits[(q+1)% N])
                qc.cx(qubits[(q+1)% N], qubits[q])
                qc.ry(param1[q+5*N],qubits[(q+1)% N])
                qc.cx(qubits[q], qubits[(q+1)% N])
            qc.barrier()
            if d==depth-1:
                for q in range(N):
                    qc.ry(param1[q+6*N],qubits[q])
                    qc.ry(param1[q+7*N],qubits[q])
                    qc.ry(param1[q+8*N],qubits[q])
            qc.barrier()
        
        return qc

    def circ(parameters):
        x=QuantumRegister(12)
        c=ClassicalRegister(8)
        circuit = QuantumCircuit(x,c)
    
        circuit=circuit.compose(ansatz(parameters),x[8:12])
        circuit.h(x[0:8]) 
        circuit.append(C_u1gate, [x[0],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u2gate, [x[1],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u3gate, [x[2],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u4gate, [x[3],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u5gate, [x[4],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u6gate, [x[5],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u7gate, [x[6],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u8gate, [x[7],x[8],x[9],x[10],x[11]]) 
        circuit &= QFT(num_qubits=8, approximation_degree=0, do_swaps=True, 
                       inverse=True, insert_barriers=False, name='qft')
        circuit.measure(x[0:8],c)       
        return circuit
    
    shots = 100000
    def cost(Parameters):
        job = backend.run(pm.run(circ(Parameters)),shots = shots).result()
        result = job.get_counts(0)
        if '00000000'not in result: 
            res = 1
        else:
            res = 1 - result['00000000']/shots
        return res
    # cost(parameters)

    def Optimizer(fun, x0, args=(), maxfev=None, 
                  reset_interval=None, eps=None, callback=None, **_):
        
        x0 = np.asarray(x0)
        recycle_z0 = None
        niter = 0
        funcalls = 0
    
        while True:
    
            idx = niter % x0.size
    
            if reset_interval > 0:
                if niter % reset_interval == 0:
                    recycle_z0 = None
    
            if recycle_z0 is None:
                z0 = fun(np.copy(x0), *args)
                funcalls += 1
            else:
                z0 = recycle_z0
    
            p = np.copy(x0)
            p[idx] = x0[idx] + np.pi / 2
            z1 = fun(p, *args)
            funcalls += 1
    
            p = np.copy(x0)
            p[idx] = x0[idx] - np.pi / 2
            z3 = fun(p, *args)
            funcalls += 1
    
            z2 = z1 + z3 - z0
            c = (z1 + z3) / 2
            a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
            b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
            b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
            x0[idx] = b
            recycle_z0 = c - a
            if callback is not None:
                callback(np.copy(x0))
            if funcalls >= maxfev:
                break
            niter += 1
        # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
        #                       nfev=funcalls, success=(niter > 1))
    
    def save(Parameters):
        global Cost,Params
        Cost.append(cost(Parameters))
        Params.append(Parameters)
        # print(cost(Parameters))
    Cost = []
    Params = []
    Optimizer(cost, Parameters, args=(), maxfev = 4000, 
              reset_interval = 32, eps=1e-32, callback=save)

    e = []
    F = []
    norm_e = []
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)

    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    RMSE.append(Res)
print(RMSE)

[0.014839997829696399, 0.02054433707435863, 0.01164370209357692, 0.017465209286317986, 0.01681587229223847, 0.016845021805981027, 0.01346248267759888, 0.016256154024131116, 0.02041344048229417, 0.019506101790114933, 0.015769545533714603, 0.013228832580753901, 0.016598307014629193, 0.020821374587467342, 0.014615719053023108, 0.01691096867092411, 0.017452566515021806, 0.013770339332064788, 0.012977029386272751, 0.012461669123514601, 0.018512175672263532, 0.018425433003024932, 0.01980534848786158, 0.01463984861548177, 0.016864222542461456, 0.011601769164322076, 0.016042477138905463, 0.018663072037514026, 0.019252547239446492, 0.019286159435332406, 0.01870985513207969, 0.019782112854867594, 0.014020716717777467, 0.012991363759283586, 0.019395435412233653, 0.012370381119634226, 0.020211681301139046, 0.015428426055286944, 0.014175494368675122, 0.014594521423541392, 0.019154023811916977, 0.01304257607762057, 0.02066250504063975, 0.02072900429165616, 0.019371929913090863, 0.012351086610187119,

In [10]:
print(np.mean(RMSE))

0.016474871455980946
